*0.1 Python for GenAI*

# streaming iterators

**The situation.** Your support chatbot's answers take 3–4 seconds to write. Users see an empty box for 3 seconds, decide it is broken, and click "send" again — doubling the load. The answers were fine. The wait was the problem.

**The fix: show words as they are written.** The model produces its answer one small piece (a *token*, roughly a word) at a time. Without streaming, the server collects all the pieces and sends the finished answer. With streaming, it sends each piece the moment it exists. The first words appear after about half a second, and the answer reads itself out.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**A generator that yields the pieces.** The same idea as the previous item — hand out one piece, pause, continue — but the pieces come from the network as the model writes.

In [2]:
import time
from collections.abc import Iterator

from openai import OpenAI

client = OpenAI(timeout=30)


def stream_answer(question: str) -> Iterator[str]:
    with client.chat.completions.stream(
        model=MODEL, messages=[{"role": "user", "content": question}], temperature=0, max_tokens=80
    ) as stream:
        for event in stream:
            if event.type == "content.delta":
                yield event.delta  # one piece of the answer, as soon as it arrives


started = time.perf_counter()
first_piece_at = None
answer = ""
for piece in stream_answer("Explain in two sentences how I reset my password."):
    if first_piece_at is None:
        first_piece_at = time.perf_counter() - started
    answer += piece
    print(piece, end="", flush=True)
finished_at = time.perf_counter() - started
print(f"\n\nfirst piece after {first_piece_at:.2f} s · finished after {finished_at:.2f} s")
assert first_piece_at < finished_at

To

 reset

 your

 password

,

...

 a

 new

 password

.



first piece after 1.18 s · finished after 1.50 s


**Reading the output.** The first piece arrived well before the last one. With a normal call, the user would have seen nothing until the "finished" time. Same model, same answer, same total time — but the wait *feels* short because something is happening.

```
without streaming   [ ········· nothing ········· ] ──▶ whole answer at the end
with streaming      To  reset  your  password ,  go … ──▶ first word early, last word at the same time
```

**The rule to remember.** Stream whenever a person is watching the answer appear. Never in background jobs — nobody is watching, and streaming only adds code.

| Use it when | Don't when | Instead use |
|---|---|---|
| chat, voice, any screen a person is looking at | batch jobs and pipelines | a normal call |

**Watch out**
- If the user closes the tab, stop reading and close the stream. The provider keeps generating — and billing — until you do.
- Every hop between the model and the browser must pass pieces through. One proxy that collects the whole response turns streaming back into a 3-second wait.
- The token count arrives in the last piece. Cost tracking must read that final event.